# 12a — Optimization Algorithms (numpy)

Corresponds to: Note 12 — Optimization Algorithms: SGD, Momentum, RMSProp, Adam

**Libraries introduced in this notebook:** No new libraries — numpy only, introduced in Note 03.

**What this notebook demonstrates:**
1. SGD with momentum — implemented from scratch, worked example verified against Note 12 Part 2
2. RMSProp — implemented from scratch, worked example verified against Note 12 Part 3
3. Adam with bias correction — implemented from scratch, worked example verified against Note 12 Part 4
4. Optimizer comparison — all three on the same quadratic loss surface

In [ ]:
import numpy as np

## Part 1: SGD with Momentum

Momentum accumulates an exponential moving average of past gradients:

    vₜ = β · vₜ₋₁ + (1 − β) · g
    w  = w − η · vₜ

We implement this and reproduce the worked example from Note 12 Part 2:
w₀ = 1.0, η = 0.1, β = 0.9, gradients = [0.8, 0.6, 0.7].

In [ ]:
def momentum_step(w, v, g, eta, beta):
    v_new = beta * v + (1 - beta) * g
    w_new = w - eta * v_new
    return w_new, v_new

w, v   = 1.0, 0.0
eta    = 0.1
beta   = 0.9
grads  = [0.8, 0.6, 0.7]

print('SGD with Momentum (β=0.9, η=0.1):')
print(f'  Step 0:  w = {w:.4f}')
for t, g in enumerate(grads, 1):
    w, v = momentum_step(w, v, g, eta, beta)
    print(f'  Step {t}:  v = {v:.4f},  w = {w:.4f}')

print()
print('Expected from Note 12 Part 2:')
print('  Step 1:  v = 0.0800,  w = 0.9920')
print('  Step 2:  v = 0.1320,  w = 0.9788')
print('  Step 3:  v = 0.1888,  w = 0.9599')

## Part 2: RMSProp

RMSProp adapts the learning rate per parameter using the root mean square of recent gradients:

    sₜ = β · sₜ₋₁ + (1 − β) · g²
    w  = w − (η / √(sₜ + ε)) · g

We reproduce the worked example from Note 12 Part 3:
w₀ = 1.0, η = 0.1, β = 0.9, ε = 1e-8, gradients = [0.8, 0.6, 0.7].

In [ ]:
def rmsprop_step(w, s, g, eta, beta, eps=1e-8):
    s_new = beta * s + (1 - beta) * g ** 2
    w_new = w - (eta / np.sqrt(s_new + eps)) * g
    return w_new, s_new

w, s   = 1.0, 0.0
eta    = 0.1
beta   = 0.9
grads  = [0.8, 0.6, 0.7]

print('RMSProp (β=0.9, η=0.1):')
print(f'  Step 0:  w = {w:.4f}')
for t, g in enumerate(grads, 1):
    w, s = rmsprop_step(w, s, g, eta, beta)
    print(f'  Step {t}:  s = {s:.6f},  w = {w:.4f}')

print()
print('Expected from Note 12 Part 3:')
print('  Step 1:  s = 0.064000,  w = 0.6838')
print('  Step 2:  s = 0.093600,  w = 0.4877')
print('  Step 3:  s = 0.133240,  w = 0.2959')

## Part 3: Adam with Bias Correction

Adam maintains two moment estimates with bias correction:

    mₜ = β₁ · mₜ₋₁ + (1 − β₁) · g          ← first moment
    vₜ = β₂ · vₜ₋₁ + (1 − β₂) · g²         ← second moment
    m̂ₜ = mₜ / (1 − β₁ᵗ)                    ← bias correction
    v̂ₜ = vₜ / (1 − β₂ᵗ)
    w   = w − η · m̂ₜ / (√v̂ₜ + ε)

We reproduce the worked example from Note 12 Part 4:
w₀ = 1.0, η = 0.1, β₁ = 0.9, β₂ = 0.999, ε = 1e-8, gradients = [0.8, 0.6, 0.7].

In [ ]:
def adam_step(w, m, v, g, t, eta, beta1, beta2, eps=1e-8):
    m_new = beta1 * m + (1 - beta1) * g
    v_new = beta2 * v + (1 - beta2) * g ** 2
    m_hat = m_new / (1 - beta1 ** t)
    v_hat = v_new / (1 - beta2 ** t)
    w_new = w - eta * m_hat / (np.sqrt(v_hat) + eps)
    return w_new, m_new, v_new

w, m, v = 1.0, 0.0, 0.0
eta     = 0.1
beta1   = 0.9
beta2   = 0.999
grads   = [0.8, 0.6, 0.7]

print('Adam (β₁=0.9, β₂=0.999, η=0.1):')
print(f'  Step 0:  w = {w:.4f}')
for t, g in enumerate(grads, 1):
    m_prev, v_prev = m, v
    w, m, v = adam_step(w, m, v, g, t, eta, beta1, beta2)
    m_hat = m / (1 - beta1 ** t)
    v_hat = v / (1 - beta2 ** t)
    print(f'  Step {t}:  m={m:.4f}, v={v:.6f}, m̂={m_hat:.4f}, v̂={v_hat:.4f}, w={w:.4f}')

print()
print('Expected from Note 12 Part 4:')
print('  Step 1:  w = 0.9000')
print('  Step 2:  w = 0.8018')
print('  Step 3:  w = 0.7029')

## Part 4: Optimizer Comparison

We compare all three optimizers on a simple quadratic loss L(w) = w² with analytic gradient g = 2w.
Starting at w₀ = 5.0, the minimum is at w = 0.

We run 50 steps and record w at each step to see how quickly each optimizer converges.

In [ ]:
def run_optimizer(optimizer, w0=5.0, n_steps=50, eta=0.1):
    w = w0
    state = {'v': 0.0, 's': 0.0, 'm': 0.0}   # shared state dict
    trajectory = [w]
    for t in range(1, n_steps + 1):
        g = 2 * w   # gradient of L = w²
        w, state = optimizer(w, g, t, state, eta)
        trajectory.append(w)
    return trajectory

def opt_momentum(w, g, t, state, eta, beta=0.9):
    state['v'] = beta * state['v'] + (1 - beta) * g
    return w - eta * state['v'], state

def opt_rmsprop(w, g, t, state, eta, beta=0.9, eps=1e-8):
    state['s'] = beta * state['s'] + (1 - beta) * g ** 2
    return w - (eta / np.sqrt(state['s'] + eps)) * g, state

def opt_adam(w, g, t, state, eta, beta1=0.9, beta2=0.999, eps=1e-8):
    state['m'] = beta1 * state['m'] + (1 - beta1) * g
    state['s'] = beta2 * state['s'] + (1 - beta2) * g ** 2
    m_hat = state['m'] / (1 - beta1 ** t)
    v_hat = state['s'] / (1 - beta2 ** t)
    return w - eta * m_hat / (np.sqrt(v_hat) + eps), state

traj_momentum = run_optimizer(opt_momentum)
traj_rmsprop  = run_optimizer(opt_rmsprop)
traj_adam     = run_optimizer(opt_adam)

print(f"{'Step':<8} {'Momentum':>12} {'RMSProp':>12} {'Adam':>12}")
print("-" * 48)
for step in [0, 1, 5, 10, 20, 50]:
    print(f"{step:<8} {traj_momentum[step]:>12.4f} {traj_rmsprop[step]:>12.4f} {traj_adam[step]:>12.4f}")

print()
print(f'Final w (step 50) — minimum is at w=0:')
print(f'  Momentum: {traj_momentum[-1]:.6f}')
print(f'  RMSProp:  {traj_rmsprop[-1]:.6f}')
print(f'  Adam:     {traj_adam[-1]:.6f}')

## Summary

This notebook demonstrated:

1. **SGD with momentum** — velocity accumulates past gradients with β=0.9. Worked example matches Note 12 Part 2 exactly. Consistent gradients build velocity; alternating gradients cancel.
2. **RMSProp** — running average of squared gradients scales the effective learning rate per parameter. Worked example matches Note 12 Part 3. Takes larger steps early when s is small.
3. **Adam** — combines momentum and RMSProp with bias correction. Bias correction at step 1 restores the true gradient scale (m̂ ≈ g, v̂ ≈ g²), giving an effective step of approximately η. Worked example matches Note 12 Part 4.
4. **Optimizer comparison** — on a quadratic loss, all three converge to w=0. RMSProp and Adam converge faster than momentum due to adaptive learning rates.